# 06 · ADX filter sensitivity (research IS only)

Other test — **not** a STAR change. Compares `trend_mode=off` vs `adx_veto` and asks
whether ADX>25 at entry is associated with lucky trending winners, unlucky losers, or both.

Does not reopen H-007. No STAR write.

## 0. Imports & Config

In [ ]:
from __future__ import annotations

import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from IPython.display import display

from backtest.s2_coint.diagnosis import enrich_trades, extreme_trades
from backtest.s2_coint.report import load_star_stack, require_star
from backtest.s2_coint.research import (
    DEFAULT_STAR_STACK,
    config_from_stack,
    frozen_pairs_for_universe,
    is_end_for_stack,
    load_s1_weekly,
    load_star_panels,
    load_universe_panels,
    lookbacks_for_bar,
    overlay_kalman_hedge,
    overlay_ols_hedge,
    split_is_oos,
)
from backtest.s2_coint.tearsheet import cvar, fit_mean_abs_score
from strategies.s2_coint.engine import simulate_book
from strategies.s2_coint.metrics import corr_to_s1, metrics_from_returns_inference
from strategies.s2_coint.sizing import pair_scale_from_score

warnings.filterwarnings("ignore", category=FutureWarning)

N_EXTREME = 7
STAR_PATH = DEFAULT_STAR_STACK
stack = load_star_stack(STAR_PATH)
require_star("UNIVERSE_STAR", stack.get("UNIVERSE_STAR"))
require_star("EXIT_STAR", stack.get("EXIT_STAR"))
UNIVERSE = str(stack["UNIVERSE_STAR"])
PAIRS = list(stack.get("PAIRS_STAR") or frozen_pairs_for_universe(UNIVERSE, "1d", root=ROOT))
print("UNIVERSE", UNIVERSE)
print("PAIRS", PAIRS)
print("EXIT_STAR", stack.get("EXIT_STAR"))
print("BREAK_STAR", stack.get("BREAK_STAR"))
print("SIZE_STAR", stack.get("SIZE_STAR"))
print("TREND_STAR", stack.get("TREND_STAR"))
print("VOL_STAR", stack.get("VOL_STAR"))
print("NOTE: STAR stack is read-only in this notebook (other_tests).")

In [ ]:
def _cagr(returns: pd.Series, periods_per_year: float = 252.0) -> float:
    r = pd.to_numeric(returns, errors="coerce").fillna(0.0).astype(float)
    if r.empty:
        return float("nan")
    total = float((1.0 + r).prod())
    years = len(r) / float(periods_per_year)
    if years <= 0 or total <= 0:
        return float("nan")
    return float(total ** (1.0 / years) - 1.0)


def arm_metrics(returns: pd.Series, s1: pd.Series | None = None) -> dict:
    r = pd.to_numeric(returns, errors="coerce").fillna(0.0).astype(float)
    r.index = pd.to_datetime(r.index)
    m = metrics_from_returns_inference(r, periods_per_year=252.0)
    cagr = _cagr(r)
    mdd = float(m.get("max_drawdown", float("nan")))
    calmar = float(cagr / abs(mdd)) if np.isfinite(cagr) and np.isfinite(mdd) and mdd != 0 else float("nan")
    return {
        "ann_sharpe": m.get("ann_sharpe", float("nan")),
        "max_drawdown": mdd,
        "calmar": calmar,
        "cagr": cagr,
        "skew": m.get("skew", float("nan")),
        "excess_kurtosis": m.get("excess_kurtosis", float("nan")),
        "cvar_5pct": cvar(r, alpha=0.05),
        "corr_to_s1": corr_to_s1(r, s1 if s1 is not None else s1_weekly),
        "n_days": m.get("n_days", 0),
    }


def collect_trades(book, panel: pd.DataFrame) -> pd.DataFrame:
    frames = []
    for pid, res in book.pair_results.items():
        if res.trades is None or res.trades.empty:
            continue
        t = res.trades.copy()
        t["pair_id"] = str(pid)
        frames.append(enrich_trades(t, panel, res.returns))
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def show_extreme(trades: pd.DataFrame, n: int = N_EXTREME, title: str = "") -> None:
    best, worst = extreme_trades(trades, n=n)
    cols = [
        "pair_id", "side_label", "entry_date", "exit_date", "hold_bars",
        "exit_reason", "pnl_pct", "z_entry", "z_exit", "adf_entry", "adf_exit",
    ]
    if title:
        print(title)
    print(f"=== Top {n} trades ===")
    display(best[cols] if not best.empty else best)
    print(f"=== Bottom {n} trades ===")
    display(worst[cols] if not worst.empty else worst)


def metrics_table(rows: dict[str, dict]) -> pd.DataFrame:
    df = pd.DataFrame(rows).T
    order = [
        "ann_sharpe", "max_drawdown", "calmar", "cagr", "skew",
        "excess_kurtosis", "cvar_5pct", "corr_to_s1", "n_days",
    ]
    cols = [c for c in order if c in df.columns] + [c for c in df.columns if c not in order]
    return df[cols]

from dataclasses import replace
from strategies.s2_coint.spread_ohlc import attach_spread_indicators

In [ ]:
bar = str(stack.get("BAR_STAR") or "1d")
lb = lookbacks_for_bar(bar)
# Prefer cached STAR panels when present; else overlay hedge on the research-IS train panel only.
try:
    train_star, _full_star, _manifest = load_star_panels(
        universe=UNIVERSE, bar=bar, pair_ids=PAIRS, root=ROOT
    )
    is_end = is_end_for_stack(stack, train_star)
    is_raw, _oos_unused = split_is_oos(train_star, is_end=is_end)
    del _oos_unused
    panel_src = "cached star train"
except (FileNotFoundError, ValueError) as exc:
    print("star panels unavailable (", type(exc).__name__, ") — using universe train panel")
    train, _full = load_universe_panels(UNIVERSE, bar, PAIRS, root=ROOT)
    is_end = is_end_for_stack(stack, train)
    is_raw, _oos_unused = split_is_oos(train, is_end=is_end)
    del _oos_unused
    panel_src = "universe train"

hedge = str(stack.get("HEDGE_STAR") or "ols")
# Skip re-overlay when star cache already has z / adf columns.
need_overlay = "z" not in is_raw.columns or "adf_pvalue" not in is_raw.columns
if need_overlay:
    if hedge == "kalman":
        is_panel = overlay_kalman_hedge(
            is_raw,
            z_window=lb["z_window"],
            hl_window=lb["hl_window"],
            adf_window=lb["adf_window"],
        )
    else:
        is_panel = overlay_ols_hedge(
            is_raw,
            ols_window=lb["ols_window"],
            z_window=lb["z_window"],
            hl_window=lb["hl_window"],
            adf_window=lb["adf_window"],
        )
else:
    is_panel = is_raw.copy()

is_panel = is_panel.loc[is_panel["pair_id"].astype(str).isin(PAIRS)].copy()
is_panel["date"] = pd.to_datetime(is_panel["date"])

mean_abs = fit_mean_abs_score(is_panel, score_column="z")
s1_weekly = load_s1_weekly(ROOT)
cfg_star = config_from_stack(stack)
print("panel_src", panel_src, "need_overlay", need_overlay)
print("bar", bar, "is_end", is_end)
print("IS rows", len(is_panel), "pairs", is_panel["pair_id"].nunique())
print("frozen IS mean(|z|)", round(mean_abs, 4))
print("cfg", cfg_star)

In [ ]:
panel_ohlc = attach_spread_indicators(is_panel, atr_window=int(cfg_star.atr_window))
assert "adx_spread" in panel_ohlc.columns, "adx_spread missing — TA-Lib required for this notebook"
print("adx_spread finite share", float(np.isfinite(panel_ohlc["adx_spread"]).mean()))

## 1. Baseline vs ADX Veto

In [ ]:
results = {}
trades_by = {}
books = {}

for name, mode in [("off", "off"), ("adx_veto", "adx_veto")]:
    cfg = replace(cfg_star, trend_mode=mode)
    book = simulate_book(panel_ohlc, cfg, mean_abs_score=mean_abs)
    books[name] = book
    results[name] = arm_metrics(book.returns)
    trades_by[name] = collect_trades(book, panel_ohlc)
    print(name, results[name])
    show_extreme(trades_by[name], title=f"{name} extremes")

display(metrics_table(results))

## 2. ADX Regime at Entry (baseline trades)

In [ ]:
base_trades = trades_by["off"].copy()

def adx_at_signal(row) -> float:
    pid = str(row.pair_id)
    sig = pd.Timestamp(row.signal_date) if pd.notna(row.signal_date) else pd.NaT
    g = panel_ohlc.loc[panel_ohlc["pair_id"].astype(str) == pid]
    if pd.isna(sig) or g.empty:
        return float("nan")
    hit = g.loc[pd.to_datetime(g["date"]) == sig]
    if hit.empty:
        hit = g.loc[pd.to_datetime(g["date"]) <= sig].tail(1)
    if hit.empty or "adx_spread" not in hit.columns:
        return float("nan")
    return float(hit["adx_spread"].iloc[0])

base_trades["adx_entry"] = [adx_at_signal(r) for r in base_trades.itertuples(index=False)]
base_trades["adx_gt_25"] = base_trades["adx_entry"] > 25.0
print("ADX entry describe")
display(base_trades["adx_entry"].describe())
print("bucket counts")
display(base_trades["adx_gt_25"].value_counts(dropna=False))

for flag, label in [(False, "adx<=25"), (True, "adx>25")]:
    sub = base_trades.loc[base_trades["adx_gt_25"] == flag, "pnl_pct"].dropna()
    print(f"=== {label} n={len(sub)} ===")
    if len(sub):
        print(sub.describe())
        print("tail min/max", float(sub.min()), float(sub.max()))

## 3. Trending Trade Anatomy

In [ ]:
best, worst = extreme_trades(base_trades, n=N_EXTREME)
for label, frame in [("winners", best), ("losers", worst)]:
    f = frame.copy()
    f["adx_entry"] = [adx_at_signal(r) for r in f.itertuples(index=False)]
    f["would_block_adx_veto"] = f["adx_entry"] > 25.0
    print(f"=== Top/bottom anatomy ({label}) ====")
    display(f[[
        "pair_id", "side_label", "entry_date", "exit_date", "pnl_pct",
        "exit_reason", "adx_entry", "would_block_adx_veto",
    ]])

n_win_block = int((best.assign(adx=[adx_at_signal(r) for r in best.itertuples(index=False)])["adx"] > 25).sum()) if not best.empty else 0
# recompute cleanly
best_adx = best.copy()
best_adx["adx_entry"] = [adx_at_signal(r) for r in best_adx.itertuples(index=False)]
worst_adx = worst.copy()
worst_adx["adx_entry"] = [adx_at_signal(r) for r in worst_adx.itertuples(index=False)]
print("top winners that would be blocked:", int((best_adx["adx_entry"] > 25).sum()), "/", len(best_adx))
print("bottom losers that would be blocked:", int((worst_adx["adx_entry"] > 25).sum()), "/", len(worst_adx))

## 4. Return Distribution Comparison

In [ ]:
r_off = books["off"].returns.astype(float)
r_adx = books["adx_veto"].returns.astype(float)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].hist(r_off.dropna(), bins=40, alpha=0.55, label="off", color="#1f4e79", density=True)
axes[0].hist(r_adx.dropna(), bins=40, alpha=0.55, label="adx_veto", color="#c62828", density=True)
axes[0].legend()
axes[0].set_title("Daily return density")
axes[0].grid(True, alpha=0.3)

# QQ vs each other (sorted)
a = np.sort(r_off.fillna(0.0).to_numpy())
b = np.sort(r_adx.reindex(r_off.index).fillna(0.0).to_numpy())
n = min(len(a), len(b))
axes[1].scatter(a[:n], b[:n], s=8, alpha=0.5)
lim = [min(a.min(), b.min()), max(a.max(), b.max())]
axes[1].plot(lim, lim, "k--", lw=1)
axes[1].set_xlabel("off quantiles")
axes[1].set_ylabel("adx_veto quantiles")
axes[1].set_title("QQ off vs adx_veto")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

display(metrics_table(results)[["ann_sharpe", "skew", "excess_kurtosis", "cvar_5pct", "max_drawdown", "calmar", "corr_to_s1"]])

## 5. Summary

Does ADX veto remove lucky trending winners, unlucky losers, or both?
Net effect on excess kurtosis / Sharpe / corr_to_s1?
Worth a later formal H-007 rematch?